# 🏛️ Aplicativo MCM - Classificador Fiscal, NCM & Precificação (Google Colab Launcher)

Este notebook executa o **Aplicativo MCM** diretamente em uma máquina virtual do Google Colab com **arquitetura híbrida anti-disco efêmero** (código no SSD ultra-rápido da VM e bases fiscais em `/content/drive/MyDrive/Aplicativo_MCM_Data`).

### Destaques:
- **Segurança Reforçada:** A rota de atualização atômica opera com `ENABLE_UPDATE_ROUTE=false` por padrão quando exposta à internet.
- **Túnel HTTPS Automático:** Utiliza `cloudflared` oficial para prover um endereço web seguro `https://*.trycloudflare.com`.
- **Validação Integrada:** Célula inclusa para testar o Barramento de Webhook Fiscal (`/api/webhook/fiscal`).

--- 
## 📦 Célula 1: Montagem do Google Drive e Setup Híbrido (Anti-Disco Efêmero)
Esta etapa monta o Google Drive, garante o clone do repositório no SSD da VM e conecta as pastas `public/data` e `rules` ao Google Drive para reter históricos de parametrizações e versões fiscais.

In [ ]:
# 1. Montagem do Google Drive
import os
import shutil
from google.colab import drive

print("==> [1/4] Montando Google Drive...")
drive.mount('/content/drive')

# 2. Criação das pastas de dados persistentes no Google Drive
DRIVE_DIR = '/content/drive/MyDrive/Aplicativo_MCM_Data'
DRIVE_DATA = os.path.join(DRIVE_DIR, 'public_data')
DRIVE_RULES = os.path.join(DRIVE_DIR, 'rules')
os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DRIVE_RULES, exist_ok=True)

# 3. Clone ou atualização do repositório no SSD local da VM (/content)
APP_DIR = '/content/Aplicativo_MCM-'
REPO_URL = 'https://github.com/SEU_USUARIO/Aplicativo_MCM-.git'  # Ajuste com a URL oficial do seu repositório

if not os.path.exists(APP_DIR):
    print("==> [2/4] Clonando repositório para o SSD local...")
    # Caso esteja subindo via upload direto de pasta compactada ou clone:
    !git clone $REPO_URL $APP_DIR || mkdir -p $APP_DIR
else:
    print("==> [2/4] Repositório existente no SSD local.")

# 4. Vinculação das pastas de dados persistentes via Symlink (Anti-Disco Efêmero)
print("==> [3/4] Sincronizando e vinculando dados ao Google Drive...")
os.chdir(APP_DIR)

# Se o Drive estiver vazio, copia os dados iniciais do repositório para ele
if os.path.exists('public/data') and not os.listdir(DRIVE_DATA):
    print("    Copiando public/data inicial para o Drive...")
    shutil.copytree('public/data', DRIVE_DATA, dirs_exist_ok=True)
if os.path.exists('rules') and not os.listdir(DRIVE_RULES):
    print("    Copiando rules inicial para o Drive...")
    shutil.copytree('rules', DRIVE_RULES, dirs_exist_ok=True)

# Cria links simbólicos para que o Next.js leia e grave no Google Drive
if os.path.exists('public/data') and not os.path.islink('public/data'):
    shutil.rmtree('public/data')
    os.symlink(DRIVE_DATA, 'public/data')

if os.path.exists('rules') and not os.path.islink('rules'):
    shutil.rmtree('rules')
    os.symlink(DRIVE_RULES, 'rules')

# 5. Instalação de dependências do Node.js
print("==> [4/4] Instalando dependências (npm ci)... ")
!npm ci --prefer-offline --no-audit
print("\n[OK] Setup Híbrido Concluído com Sucesso!")

--- 
## ⚖️ Célula 2: Atualização Legal e Pipeline Sob Demanda (Opcional)
Executa os scripts Python determinísticos para verificar se há novidades no marco legal ou tabelas oficiais do Siscomex/CONFAZ, salvando o resultado diretamente no Google Drive montado.

In [ ]:
import os
os.chdir('/content/Aplicativo_MCM-')
print("==> Verificando Marco Legal e gerando search.json atualizado...")
!python3 scripts/atualizar_marco_legal.py --check
if os.path.exists('scripts/pipeline.py'):
    !python3 scripts/pipeline.py
print("\n[OK] Marco Legal e bases atualizadas com sucesso no Google Drive!")

--- 
## 🌐 Célula 3: Inicialização do Next.js + Túnel Público HTTPS (Cloudflared)
Esta célula compila o Next.js em modo produção (`npm run build`), inicia o servidor na porta 3000 em background com `ENABLE_UPDATE_ROUTE=false` e inicializa o `cloudflared` para criar o túnel HTTPS seguro com URL pública temporária.

In [ ]:
import os
import time
import subprocess

os.chdir('/content/Aplicativo_MCM-')

# 1. Instalação do Cloudflared se não estiver presente
if not os.path.exists('/usr/local/bin/cloudflared'):
    print("==> Baixando binário oficial do Cloudflared...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null

# 2. Compilação da aplicação Next.js para produção
print("==> Compilando Next.js (npm run build)...")
!npm run build

# 3. Inicialização do servidor Next.js em background
print("==> Iniciando servidor Next.js na porta 3000 com ENABLE_UPDATE_ROUTE=false...")
env_vars = os.environ.copy()
env_vars["ENABLE_UPDATE_ROUTE"] = "false"
env_vars["NODE_ENV"] = "production"
env_vars["PORT"] = "3000"

server_process = subprocess.Popen(["npm", "start"], env=env_vars, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# 4. Abertura do túnel público HTTPS com Cloudflared
print("\n===================================================================")
print("==> Iniciando Túnel HTTPS. Copie o endereço *.trycloudflare.com abaixo:")
print("===================================================================\n")
!cloudflared tunnel --url http://localhost:3000

--- 
## 📡 Célula 4: Simulador de Chamada ao Webhook Fiscal (`/api/webhook/fiscal`)
Demonstra a integração externa com o barramento fiscal determinístico via requisição autenticada com Bearer Token e payload de produtos para parametrização automática em lote.

In [ ]:
import requests
import json

# Altere para localhost:3000 se executado internamente, ou para a URL do seu túnel Cloudflare
URL_API = "http://localhost:3000/api/webhook/fiscal"
TOKEN = "mcm_dev_test_token_2026"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "modo": "lote_produtos",
    "uf_origem": "SP",
    "uf_destino": "PR",
    "regime_destinatario": "real",
    "destinacao": "comercializacao",
    "produtos": [
        {
            "codigo": "PROD-COLH",
            "descricao": "Colheitadeira de Graos com plataforma de corte",
            "ncm": "84335100",
            "cfop": "6102",
            "valor_unitario": 750000.0
        },
        {
            "codigo": "PROD-ARROZ",
            "descricao": "Arroz beneficiado polido tipo 1",
            "ncm": "10063021",
            "cfop": "5102",
            "valor_unitario": 5.50
        }
    ]
}

print("==> Enviando lote para auditoria e parametrização no Webhook Fiscal...")
try:
    resp = requests.post(URL_API, json=payload, headers=headers, timeout=10)
    print(f"Status Code: {resp.status_code}")
    print("Resposta formatada:")
    print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
except Exception as e:
    print(f"Falha ao consultar API: {e}")